In [ ]:
import numpy as np
import pandas as pd


def minimale_lengte_filter(binair_signaal, min_samples):
    """
    Houd alleen stukken over waar minstens min_samples opeenvolgende 1'en zitten.

    Parameters
    ----------
    binair_signaal : array-like
        Binaire lijst of array met 0 en 1.
    min_samples : int
        Minimale lengte van een artefact in aantal samples.

    Returns
    -------
    gefilterd : np.ndarray
        Binaire array waarin te korte stukken verwijderd zijn.
    """
    binair_signaal = np.asarray(binair_signaal, dtype=int)
    gefilterd = np.zeros_like(binair_signaal)

    teller = 0
    for i in range(len(binair_signaal)):
        if binair_signaal[i] == 1:
            teller += 1
        else:
            if teller >= min_samples:
                gefilterd[i - teller:i] = 1
            teller = 0

    if teller >= min_samples:
        gefilterd[len(binair_signaal) - teller:len(binair_signaal)] = 1

    return gefilterd


def vind_start_eindtijden(binair_signaal, t, min_tussenruimte=2.0):
    """
    Zet een binair signaal om naar geldige start- en eindtijden.

    Parameters
    ----------
    binair_signaal : array-like
        Binaire array met 0 en 1.
    t : array-like
        Tijdvector.
    min_tussenruimte : float
        Minimale tijd tussen twee artefacten om ze als apart te zien.

    Returns
    -------
    valid_start_tijden : np.ndarray
    valid_eind_tijden : np.ndarray
    """
    binair_signaal = np.asarray(binair_signaal, dtype=int).copy()

    if len(binair_signaal) == 0:
        return np.array([]), np.array([])

    binair_signaal[0] = 0
    binair_signaal[-1] = 0

    if np.sum(binair_signaal) == 0:
        return np.array([]), np.array([])

    diff_signaal = np.diff(binair_signaal)

    start_idx = np.where(diff_signaal == 1)[0] + 1
    eind_idx = np.where(diff_signaal == -1)[0] + 1

    start_tijden = np.asarray(t)[start_idx]
    eind_tijden = np.asarray(t)[eind_idx]

    if len(start_tijden) == 0 or len(eind_tijden) == 0:
        return np.array([]), np.array([])

    valid_start_tijden = [start_tijden[0]]
    valid_eind_tijden = [eind_tijden[0]]

    for i in range(1, len(start_tijden)):
        if start_tijden[i] - valid_start_tijden[-1] > min_tussenruimte:
            valid_start_tijden.append(start_tijden[i])
            if i < len(eind_tijden):
                valid_eind_tijden.append(eind_tijden[i])

    return np.array(valid_start_tijden), np.array(valid_eind_tijden)



def maak_artefact_dataframe(start_tijden, eind_tijden, naam_artefact, signaal):
    """
    Maak een DataFrame in het vereiste outputformaat.

    Parameters
    ----------
    start_tijden : array-like
    eind_tijden : array-like
    naam_artefact : str
    signaal : str

    Returns
    -------
    pd.DataFrame
    """
    rows = []

    for s, e in zip(start_tijden, eind_tijden):
        rows.append([
            round(float(s), 2),
            round(float(e), 2),
            naam_artefact,
            signaal
        ])

    return pd.DataFrame(
        rows,
        columns=["Starttijd", "Eindtijd", "Naam van het artefact", "Signaal"]
    )